# LiDAR BEV : entrainement et validation sur 512 frames

Apres la memorisation du notebook 05, apprendre sur 410 frames et evaluer sur 102 autres. PFN, contexte et head Car repartent de poids neufs : aucun checkpoint de memorisation n'est reutilise. Le split aleatoire par frame est disjoint et reproductible, mais pas garanti independant par sequence ou scene. Il ne s'agit toujours pas de boites 3D completes ni d'AP KITTI. Run All ne lance pas d'entrainement avec les parametres par defaut.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / 'src/rgb_lidar_fusion').is_dir() and (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the repository or notebooks directory.')
sys.path.insert(0, str(ROOT / 'src'))
from rgb_lidar_fusion.lidar_center_training import LidarCenterTrainConfig, load_lidar_center_checkpoint
from rgb_lidar_fusion.lidar_center_trainval import build_lidar_center_split
print(f'ROOT={ROOT}')


## Parametres
Le preset conserve le pilote CUDA termine a 20 epochs, batch 1 et grille de 20 cm. Le fichier JSON garde cinq epochs comme point de depart pour un nouveau run. RUN_TRAINING=False permet de relire les resultats sans relancer l'apprentissage. CUDA exige le preflight de 2048 MiB. RESUME_EXISTING recharge uniquement latest.pt ; EPOCHS est le total vise. best.pt sert a l'inspection, pas a la reprise. Ne pas reutiliser le dossier du test sur quatre frames.

In [ ]:
RUN_TRAINING = False
RESUME_EXISTING = False
DEVICE = 'cuda'
EPOCHS = 20
CONFIG_PATH = ROOT / 'configs/training/kitti_lidar_center_512.json'
OUTPUT_DIR = ROOT / 'results/training/kitti_lidar_center_512'
CHECKPOINT_PATH = OUTPUT_DIR / 'checkpoints/latest.pt'
SELECTED_CHECKPOINT = OUTPUT_DIR / 'checkpoints/best.pt'
VALIDATION_SAMPLE_POSITION = 79
values = json.loads(CONFIG_PATH.read_text())
values.update(data_root=str((ROOT / values['data_root']).resolve()),
              output_dir=str(OUTPUT_DIR), device=DEVICE, epochs=EPOCHS)
config = LidarCenterTrainConfig.from_mapping(values)
print(f'run={RUN_TRAINING} resume={RESUME_EXISTING} device={DEVICE} total_epochs={EPOCHS}')


## Verifier le split avant apprentissage
Verifier les quatre fichiers de chaque ID et leurs empreintes sur CPU. Seuls les IDs et les hashes sont conserves ici, pas les 512 nuages prepares. Le runner recree et controle ce meme split, puis le sauvegarde dans split.json et les checkpoints. Les frames sans Car ne sont pas retirees : elles supervisent les faux positifs. Les regions entierement inconnues sont ignorees et comptees. Les labels manquants sont des erreurs.

In [ ]:
if Path(config.data_root).is_dir():
    planned_split = build_lidar_center_split(config)
    train_ids, validation_ids = planned_split['train_ids'], planned_split['validation_ids']
    overlap = set(train_ids) & set(validation_ids)
    assert not overlap
    print(f'train={len(train_ids)} validation={len(validation_ids)} overlap={len(overlap)}')
    print('train preview:', train_ids[:10])
    print('validation preview:', validation_ids[:10])
    print(planned_split['limitation'])
else:
    print(f'KITTI missing at {config.data_root}; see data/README.md.')


## Entrainement explicite dans un processus separe
Mettre RUN_TRAINING=True pour lancer. Les evaluations train et validation utilisent eval() et no_grad() ; seuls les IDs train passent dans optimizer.step(). Les donnees sont preparees frame par frame, sans cache des 512 tenseurs paddes. Des messages de progression couvrent aussi le pretraitement et les evaluations. Chaque epoch terminee sauvegarde latest.pt ; best.pt minimise val_loss, y compris le point de depart a l'epoch 0. Un echec CUDA n'est jamais relance automatiquement.

In [ ]:
if RUN_TRAINING:
    command = [sys.executable, str(ROOT / 'scripts/train_lidar_center_split.py'),
               '--config', str(CONFIG_PATH), '--device', DEVICE, '--epochs', str(EPOCHS),
               '--output-dir', str(OUTPUT_DIR)]
    if RESUME_EXISTING:
        if not CHECKPOINT_PATH.is_file():
            raise FileNotFoundError(CHECKPOINT_PATH)
        command += ['--resume-from', str(CHECKPOINT_PATH)]
    env = dict(os.environ, CUDA_LAUNCH_BLOCKING='1', PYTHONUNBUFFERED='1')
    with subprocess.Popen(command, cwd=ROOT, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        try:
            for line in process.stdout:
                try:
                    event = json.loads(line)
                except json.JSONDecodeError:
                    print(line.rstrip(), flush=True)
                    continue
                if event['event'] == 'evaluation':
                    print(f"epoch={event['epoch']} step={event['step']} "
                          f"train_eval_loss={event['train_eval_loss']:.4f} val_loss={event['val_loss']:.4f} "
                          f"val_TP/FP/FN={event['val_tp']}/{event['val_fp']}/{event['val_fn']} "
                          f"val_precision={event['val_precision']} val_recall={event['val_recall']} "
                          f"best_epoch={event['best_epoch']}", flush=True)
                else:
                    print(event, flush=True)
            if process.wait() != 0:
                raise RuntimeError(f'Training process failed with code {process.returncode}.')
        except BaseException:
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
            raise
else:
    print('Training disabled. Set RUN_TRAINING=True after reviewing the split.')


## Comparer les courbes train et validation
Les deux losses tracees sont mesurees en mode eval, avec les memes poids a la fin de chaque epoch. train_loss, enregistree separement, vient des mises a jour en mode train et de ses statistiques BatchNorm. Les losses sont moyennes par frame supervisee. Precision/rappel : seuil 0.3, maxima locaux, top-k 50, appariement glouton par score a 1 m dans le domaine annote. L'ecart metrique porte seulement sur les centres apparies : regarder aussi TP/FP/FN. None signifie absence de centres cibles ou d'appariements, selon la metrique ; jamais une erreur nulle. best.pt selectionne sur validation n'est pas un resultat sur un test independant.

In [ ]:
metrics_path = OUTPUT_DIR / 'metrics.json'
metrics = json.loads(metrics_path.read_text()) if metrics_path.is_file() else []
if metrics:
    epochs = [row['epoch'] for row in metrics]
    fig, axes = plt.subplots(2, 2, figsize=(12, 7))
    panels = [(axes[0, 0], 'Loss in eval mode', 'Loss', [('train_eval_loss', 'train'), ('val_loss', 'validation')]),
              (axes[0, 1], 'Center precision', 'Fraction', [('train_precision', 'train'), ('val_precision', 'validation')]),
              (axes[1, 0], 'Center recall', 'Fraction', [('train_recall', 'train'), ('val_recall', 'validation')]),
              (axes[1, 1], 'Matched centers only', 'Mean XY error (m)', [('train_matched_mean_distance_m', 'train'), ('val_matched_mean_distance_m', 'validation')])]
    for ax, title, ylabel, series in panels:
        for key, label in series:
            ax.plot(epochs, [row[key] for row in metrics], marker='o', label=label)
        ax.set(xlabel='Epoch', ylabel=ylabel, title=title)
        ax.legend()
        ax.grid(alpha=0.3)
    axes[0, 0].set_yscale('log')
    axes[0, 1].set_ylim(-0.05, 1.05)
    axes[1, 0].set_ylim(-0.05, 1.05)
    fig.tight_layout()
    display(fig)
    plt.close(fig)
    display({key: value for key, value in metrics[-1].items() if 'per_frame' not in key})
else:
    print('No metrics yet. No training is started by this cell.')


## Inspecter une frame de validation
Charger best.pt sur CPU, puis choisir VALIDATION_SAMPLE_POSITION dans la liste de validation sauvegardee avec le run. Ce n'est pas un identifiant KITTI direct. La cellule refuse tout chevauchement avec le train. Les annotations servent aux cibles et au domaine d'evaluation, pas aux features d'entree. Revue du pilote : position 0 / 000009, centre retrouve ; 28 / 000148, voiture manquee ; 79 / 000388, faux positif. Le meilleur checkpoint selon val_loss est l'epoch 9, pas la derniere epoch 20. Voir le [bilan et les limites du pilote](../docs/workstreams/lidar-center-diagnostic.md#6-pilot-results-and-prototype-closure).

In [ ]:
if SELECTED_CHECKPOINT.is_file():
    import torch
    from rgb_lidar_fusion.lidar_bev_analysis import analyze_lidar_bev_sample, plot_lidar_bev_sample
    torch.set_num_threads(2)
    model, saved_config, saved_metrics = load_lidar_center_checkpoint(SELECTED_CHECKPOINT)
    saved_split = json.loads((SELECTED_CHECKPOINT.parent.parent / 'split.json').read_text())
    assert not set(saved_split['train_ids']) & set(saved_split['validation_ids'])
    sample_id = saved_split['validation_ids'][VALIDATION_SAMPLE_POSITION]
    assert sample_id not in saved_split['train_ids']
    print(f"Validation ID={sample_id}, checkpoint epoch={saved_metrics[-1]['epoch']}")
    sample = analyze_lidar_bev_sample(
        saved_config.data_root, sample_id, grid=saved_config.grid, model=model,
        score_threshold=saved_config.score_threshold, top_k=saved_config.top_k,
        max_points_per_pillar=saved_config.max_points_per_pillar, pillar_seed=saved_config.seed)
    fig = plot_lidar_bev_sample(sample, weights_label=f"validation, epoch {saved_metrics[-1]['epoch']}")
    display(fig)
    plt.close(fig)
else:
    print('No selected checkpoint yet; no trained validation prediction to display.')
